# DCOPF Example (Data from File)
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

In [1]:
from pyomo.environ import *
import re
import time

# ─────────────────────────────────────────────────────────────────
# Base path & data file
# ─────────────────────────────────────────────────────────────────
BASE_PATH = '/Users/csab/Desktop/ECE6379_Python_Code/20_DCOPF/'
data_file = BASE_PATH + 'DCOPF_IC_e1_M2M3_data.txt'

# Fixed system parameters
BaseMW = 100
refBus = 3

In [3]:
# ─────────────────────────────────────────────────────────────────
def parse_ampl_data(filepath):
    with open(filepath, 'r') as f:
        raw_lines = f.readlines()

    # Strip inline comments, then join
    clean_lines = [re.sub(r'#.*', '', l).strip() for l in raw_lines]
    clean_lines = [l for l in clean_lines if l]
    text = ' '.join(clean_lines)

    blocks = [b.strip() for b in re.split(r';', text) if b.strip()]

    sets   = {}   # set_name  -> [indices]
    params = {}   # param_name -> {index: value}  or  {(i,j): value}

    for block in blocks:

        # ── 1D block: param: SET_NAME: col1 col2 ... := row_data ──
        m = re.match(r'param\s*:\s*(\w+)\s*:\s*([\w\s]+):=\s*(.*)', block, re.S)
        if m:
            set_name  = m.group(1).strip()
            col_names = m.group(2).split()
            tokens    = m.group(3).split()
            n_cols    = len(col_names)
            idx_list  = []
            for col in col_names:
                params.setdefault(col, {})
            it = iter(tokens)
            try:
                while True:
                    idx = int(next(it))
                    idx_list.append(idx)
                    for col in col_names:
                        params[col][idx] = float(next(it))
            except StopIteration:
                pass
            sets[set_name] = idx_list
            continue

        # ── 2D block: param NAME: col1 col2 ... := row_data ───────
        m = re.match(r'param\s+(\w+)\s*:\s*([\d\s]+):=\s*(.*)', block, re.S)
        if m:
            param_name  = m.group(1).strip()
            col_indices = [int(c) for c in m.group(2).split()]
            tokens      = m.group(3).split()
            param_dict  = {}
            it = iter(tokens)
            try:
                while True:
                    row_idx = int(next(it))
                    for col_idx in col_indices:
                        param_dict[(row_idx, col_idx)] = float(next(it))
            except StopIteration:
                pass
            params[param_name] = param_dict
            continue

    return sets, params


# ─────────────────────────────────────────────────────────────────
# Load data
# ─────────────────────────────────────────────────────────────────
sets, params = parse_ampl_data(data_file)

BUS_data    = sets['BUS']
GEN_data    = sets['GEN']
BRANCH_data = sets['BRANCH']

busLoad_data       = {int(k): v       for k, v in params['busLoad'].items()}
gen_bus_data       = {int(k): int(v)  for k, v in params['gen_bus'].items()}
gen_min_data       = {int(k): v       for k, v in params['gen_min'].items()}
gen_max_data       = {int(k): v       for k, v in params['gen_max'].items()}
gen_OpCost_data    = {int(k): v       for k, v in params['gen_OpCost'].items()}
branch_frmbus_data = {int(k): int(v)  for k, v in params['branch_frmbus'].items()}
branch_tobus_data  = {int(k): int(v)  for k, v in params['branch_tobus'].items()}
branch_x_data      = {int(k): v       for k, v in params['branch_x'].items()}
branch_rate_data   = {int(k): v       for k, v in params['branch_rate'].items()}

print('BUS    :', BUS_data)
print('GEN    :', GEN_data)
print('BRANCH :', BRANCH_data)
print('busLoad:', busLoad_data)
print('gen_bus:', gen_bus_data)
print('branch_x:', branch_x_data)

BUS    : [1, 2, 3]
GEN    : [1, 2]
BRANCH : [1, 2, 3]
busLoad: {1: 0.0, 2: 100.0, 3: 0.0}
gen_bus: {1: 1, 2: 3}
branch_x: {1: 0.1, 2: 0.1, 3: 0.2}


In [5]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.BUS    = Set(initialize=BUS_data)
model.BRANCH = Set(initialize=BRANCH_data)
model.GEN    = Set(initialize=GEN_data)

# Parameters
model.busLoad       = Param(model.BUS,    initialize=busLoad_data)
model.gen_bus       = Param(model.GEN,    initialize=gen_bus_data,       within=model.BUS)
model.gen_min       = Param(model.GEN,    initialize=gen_min_data)
model.gen_max       = Param(model.GEN,    initialize=gen_max_data)
model.gen_OpCost    = Param(model.GEN,    initialize=gen_OpCost_data)
model.branch_frmbus = Param(model.BRANCH, initialize=branch_frmbus_data, within=model.BUS)
model.branch_tobus  = Param(model.BRANCH, initialize=branch_tobus_data,  within=model.BUS)
model.branch_x      = Param(model.BRANCH, initialize=branch_x_data)
model.branch_rate   = Param(model.BRANCH, initialize=branch_rate_data)

# Decision Variables
model.G     = Var(model.GEN)
model.pk    = Var(model.BRANCH)
model.theta = Var(model.BUS)

# ── Objective ────────────────────────────────────────────────────
def obj_rule(model):
    return sum(model.gen_OpCost[g] * model.G[g] for g in model.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Constraints ──────────────────────────────────────────────────
def branchLimits_rule(model, k):
    return inequality(-model.branch_rate[k], model.pk[k], model.branch_rate[k])
model.branchLimits = Constraint(model.BRANCH, rule=branchLimits_rule)

def lineFlowEqs_rule(model, k):
    return model.pk[k] / BaseMW == (
        model.theta[model.branch_frmbus[k]] - model.theta[model.branch_tobus[k]]
    ) / model.branch_x[k]
model.lineFlowEqs = Constraint(model.BRANCH, rule=lineFlowEqs_rule)

def genLimits_rule(model, g):
    return inequality(model.gen_min[g], model.G[g], model.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

def NodalPowerBalance_rule(model, n):
    gen_power = sum(model.G[g]  for g in model.GEN    if model.gen_bus[g]       == n)
    power_in  = sum(model.pk[k] for k in model.BRANCH if model.branch_tobus[k]  == n)
    power_out = sum(model.pk[k] for k in model.BRANCH if model.branch_frmbus[k] == n)
    return gen_power + power_in - power_out == model.busLoad[n]
model.NodalPowerBalance = Constraint(model.BUS, rule=NodalPowerBalance_rule)

# Fix reference bus angle
model.theta[refBus].fix(0)

In [7]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\nG:')
for g in model.GEN:
    print(f'  G[{g}] = {model.G[g].value}')

print('\npk:')
for k in model.BRANCH:
    print(f'  pk[{k}] = {model.pk[k].value}')

print('\ntheta:')
for n in model.BUS:
    print(f'  theta[{n}] = {model.theta[n].value}')

print(f'\nTotal solve elapsed time: {solve_time:.4f} seconds')

Set parameter Username
Set parameter LicenseID to value 2708162
Academic license - for non-commercial use only - expires 2026-09-12
Read LP format model from file /var/folders/zx/xyzw1dz90lv0dgc7dt781qf80000gn/T/tmp3ffpr_t0.pyomo.lp
Reading time = 0.00 seconds
x1: 16 rows, 7 columns, 25 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  90
MIPGap  0

Optimize a model with 16 rows, 7 columns and 25 nonzeros
Model fingerprint: 0x0ce2b637
Coefficient statistics:
  Matrix range     [1e-02, 1e+01]
  Objective range  [1e+01, 2e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 1e+02]
Presolve removed 16 rows and 7 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.    